In [5]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

price = pd.read_parquet("data/module1_spot_v0.parquet")
price["ts"] = pd.to_datetime(price["ts"], utc=True)
price = price.set_index("ts").sort_index()

ledger = pd.read_parquet("data/ledger_baseline.parquet")
ledger["entry_time"] = pd.to_datetime(ledger["entry_time"], utc=True)
ledger["exit_time"] = pd.to_datetime(ledger["exit_time"], utc=True)
ledger["t0_ts"] = pd.to_datetime(ledger["t0_ts"], utc=True)
ledger["t1_ts"] = pd.to_datetime(ledger["t1_ts"], utc=True)

meta_df = pd.read_parquet("data/btc_events.parquet")
meta_df["t0_ts"] = pd.to_datetime(meta_df["t0_ts"], utc=True)
meta_df["t1_ts"] = pd.to_datetime(meta_df["t1_ts"], utc=True)

assert not ledger.duplicated(subset=["t0_ts", "t1_ts", "fold_id"]).any()
assert not meta_df.duplicated(subset=["t0_ts", "t1_ts"]).any()

ledger_full = ledger.merge(
    meta_df[
        [
            "t0_ts", "t1_ts",
            "primary_prob",
            "sigma",
            "vol_regime",
            "trend_state",
            "p0", "p1", "p2", "p3", "p4"
        ]
    ],
    on=["t0_ts", "t1_ts"],
    how="left",
    validate="one_to_one"
)

assert len(ledger_full) == len(ledger)
assert ledger_full[["primary_prob", "sigma", "vol_regime"]].isna().sum().sum() == 0
assert ledger_full["fold_id"].equals(ledger["fold_id"])

ledger_full = ledger_full.sort_values("t0_ts").reset_index(drop=True)

print(len(price), len(ledger_full), ledger_full["fold_id"].nunique())
print(ledger_full.columns)


3056 911 10
Index(['event_id', 'fold_id', 't0_ts', 't1_ts', 'entry_time', 'exit_time', 'direction', 'ret', 'cum_pnl', 'peak',
       'drawdown', 'ret_net', 'primary_prob', 'sigma', 'vol_regime', 'trend_state', 'p0', 'p1', 'p2', 'p3', 'p4'],
      dtype='object')


In [9]:
def run_policy(meta_df, price, accept_fn, size_fn=None, policy_name="P0"):
    rows = []

    for _, ev in meta_df.iterrows():

        if not accept_fn(ev):
            continue

        entry_time = ev.t0_ts + pd.Timedelta(days=1)
        exit_time  = ev.t1_ts

        if entry_time not in price.index or exit_time not in price.index:
            continue

        entry_px = price.loc[entry_time, "open"]
        exit_px  = price.loc[exit_time, "close"]

        ret = ev.direction * (exit_px - entry_px) / entry_px

        if size_fn is not None:
            ret = ret * size_fn(ev)

        rows.append({
            "fold_id": ev.fold_id,
            "policy": policy_name,
            "ret": ret,
            "exit_time": exit_time
        })

    return pd.DataFrame(rows)


In [10]:
def summarize_policy(df):
    return (
        df.groupby(["policy", "fold_id"])
          .agg(
              trades=("ret", "count"),
              mean_ret=("ret", "mean"),
              pnl_sum=("ret", "sum"),
              max_dd=("ret", lambda x: (x.cumsum() - x.cumsum().cummax()).min())
          )
          .reset_index()
    )


In [11]:
accept_0 = lambda ev: True

p0_df = run_policy(
    ledger_full,
    price,
    accept_fn=accept_0,
    policy_name="P0_baseline"
)

p0_summary = summarize_policy(p0_df)
p0_summary


,policy,fold_id,trades,mean_ret,pnl_sum,max_dd
0,P0_baseline,0,113,0.051766,5.849606,-0.465911
1,P0_baseline,1,86,-0.013888,-1.194371,-3.053484
2,P0_baseline,2,91,-0.010211,-0.929192,-2.151494
3,P0_baseline,3,80,-0.015720,-1.257611,-2.206474
4,P0_baseline,4,84,0.003869,0.324963,-1.074055
5,P0_baseline,5,84,0.005676,0.476791,-0.737094
6,P0_baseline,6,105,0.029421,3.089177,-0.336841
7,P0_baseline,7,93,0.004224,0.392796,-1.077555
8,P0_baseline,8,90,0.021346,1.921119,-0.491756
9,P0_baseline,9,85,0.007669,0.651903,-0.888646


In [17]:
ledger_full["primary_prob"].describe()


count    911.000000
mean       0.540089
std        0.011613
min        0.525531
25%        0.525531
50%        0.549342
75%        0.549342
max        0.549342
Name: primary_prob, dtype: float64

In [18]:
ledger_full["primary_prob"].quantile([0.5, 0.6, 0.7, 0.8, 0.9])


0.5    0.549342
0.6    0.549342
0.7    0.549342
0.8    0.549342
0.9    0.549342
Name: primary_prob, dtype: float64

In [19]:
accept_B1 = lambda ev: ev.vol_regime == "high"

pB1_df = run_policy(
    ledger_full,
    price,
    accept_fn=accept_B1,
    policy_name="B_high_vol"
)

pB1_summary = summarize_policy(pB1_df)
pB1_summary


,policy,fold_id,trades,mean_ret,pnl_sum,max_dd
0,B_high_vol,0,19,0.080747,1.534189,-0.181847
1,B_high_vol,1,42,-0.044051,-1.850145,-2.450608
2,B_high_vol,2,7,0.020200,0.141400,-0.095248
3,B_high_vol,3,25,-0.040992,-1.024793,-1.141534
4,B_high_vol,4,9,-0.043048,-0.387436,-0.492849
5,B_high_vol,7,2,0.060177,0.120353,0.000000


In [20]:
accept_B2 = lambda ev: (ev.vol_regime == "high") and (ev.trend_state != "flat")

pB2_df = run_policy(
    ledger_full,
    price,
    accept_fn=accept_B2,
    policy_name="B_high_vol"
)

pB2_summary = summarize_policy(pB2_df)
pB2_summary

,policy,fold_id,trades,mean_ret,pnl_sum,max_dd
0,B_high_vol,0,19,0.080747,1.534189,-0.181847
1,B_high_vol,1,42,-0.044051,-1.850145,-2.450608
2,B_high_vol,2,7,0.020200,0.141400,-0.095248
3,B_high_vol,3,25,-0.040992,-1.024793,-1.141534
4,B_high_vol,4,9,-0.043048,-0.387436,-0.492849
5,B_high_vol,7,2,0.060177,0.120353,0.000000


In [21]:
accept_B3 = lambda ev: (
    ev.vol_regime == "high"
    and (
        (ev.direction == 1 and ev.trend_state == "up") or
        (ev.direction == -1 and ev.trend_state == "down")
    )
)

pB3_df = run_policy(
    ledger_full,
    price,
    accept_fn=accept_B3,
    policy_name="B_high_vol"
)

pB3_summary = summarize_policy(pB3_df)
pB3_summary

,policy,fold_id,trades,mean_ret,pnl_sum,max_dd
0,B_high_vol,0,19,0.080747,1.534189,-0.181847
1,B_high_vol,1,42,-0.044051,-1.850145,-2.450608
2,B_high_vol,2,7,0.020200,0.141400,-0.095248
3,B_high_vol,3,25,-0.040992,-1.024793,-1.141534
4,B_high_vol,4,9,-0.043048,-0.387436,-0.492849
5,B_high_vol,7,2,0.060177,0.120353,0.000000


In [24]:
median_sigma = ledger_full["sigma"].median()

size_fn_C = lambda ev: np.clip(ev.sigma / median_sigma, 0.5, 2.0)

accept_C = lambda ev: True

pC_df = run_policy(
    ledger_full,
    price,
    accept_fn=accept_C,
    size_fn=size_fn_C,
    policy_name="C_vol_sizing"
)

pC_summary = summarize_policy(pC_df)
pC_summary


,policy,fold_id,trades,mean_ret,pnl_sum,max_dd
0,C_vol_sizing,0,113,0.068468,7.736852,-0.410530
1,C_vol_sizing,1,86,-0.030939,-2.660755,-5.343873
2,C_vol_sizing,2,91,-0.010935,-0.995103,-2.526235
3,C_vol_sizing,3,80,-0.029824,-2.385891,-3.409149
4,C_vol_sizing,4,84,-0.006302,-0.529330,-1.639427
5,C_vol_sizing,5,84,0.009580,0.804724,-0.591075
6,C_vol_sizing,6,105,0.022177,2.328592,-0.271438
7,C_vol_sizing,7,93,0.005374,0.499798,-1.248575
8,C_vol_sizing,8,90,0.021735,1.956181,-0.374040
9,C_vol_sizing,9,85,0.003047,0.259009,-0.980744


In [25]:
def run_policy_D(ledger_full, price, lookback=2):
    rows = []
    folds = sorted(ledger_full["fold_id"].unique())
    fold_pnl = {}

    for k in folds:
        if k < lookback:
            allow = True
        else:
            recent = [fold_pnl[j] for j in range(k - lookback, k)]
            allow = np.mean(recent) > 0

        fold_df = ledger_full[ledger_full["fold_id"] == k]

        if not allow:
            fold_pnl[k] = 0.0
            continue

        pnl_sum = 0.0

        for _, ev in fold_df.iterrows():
            entry_time = ev.t0_ts + pd.Timedelta(days=1)
            exit_time  = ev.t1_ts

            if entry_time not in price.index or exit_time not in price.index:
                continue

            entry_px = price.loc[entry_time, "open"]
            exit_px  = price.loc[exit_time, "close"]

            ret = ev.direction * (exit_px - entry_px) / entry_px
            pnl_sum += ret

            rows.append({
                "fold_id": k,
                "policy": "D_causal_gate",
                "ret": ret,
                "exit_time": exit_time
            })

        fold_pnl[k] = pnl_sum

    return pd.DataFrame(rows)


In [27]:
pD_df = run_policy_D(ledger_full, price, lookback=1)
pD_summary = summarize_policy(pD_df)
pD_summary


,policy,fold_id,trades,mean_ret,pnl_sum,max_dd
0,D_causal_gate,0,113,0.051766,5.849606,-0.465911
1,D_causal_gate,1,86,-0.013888,-1.194371,-3.053484
